# 8. Fuga de datos y selección de variables

Una variable produce **fuga de datos** (*data leakage*) cuando contiene información que no estaría disponible al momento de predecir o que, directa o indirectamente, define la variable objetivo. Incluirla da métricas artificialmente altas y un modelo inútil en la práctica.

Se excluyen seis columnas, igual que en la segunda entrega.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

## 8.1 `solar_aptitude` y `solar_aptitude_rounded`: definen la clase

In [ ]:
pd.crosstab(df["solar_aptitude_rounded"], df[TARGET])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for c in CLASS_ORDER:
    ax.hist(df.loc[df[TARGET] == c, "solar_aptitude"], bins=60, range=(0, 1),
            color=CLASS_COLORS[c], label=c, alpha=0.85)
ax.set_xlabel("solar_aptitude (índice continuo)")
ax.set_ylabel("Plantas")
ax.set_title("La clase es un corte del índice solar_aptitude")
ax.legend()
plt.tight_layout(); plt.show()

La clase es un **corte directo** del índice continuo `solar_aptitude`. Valores redondeados de 0 a 3 son siempre Baja, de 7 a 9 siempre Alta, y solo los niveles 4 y 6 mezclan dos clases adyacentes. Con estas variables el modelo "predeciría" la clase copiándola, por lo que **se excluyen ambas**.

## 8.2 Variables posteriores a la decisión de inversión

In [ ]:
excl = pd.DataFrame({
    "variable": ["capacity", "optimal_tilt", "pv_potential", "operational_status"],
    "motivo": [
        "Potencia instalada: se decide después de evaluar el sitio.",
        "Ángulo de diseño de los paneles: parámetro de ingeniería del proyecto, posterior a la elección del sitio.",
        "Producción esperada por kWp: resultado del diseño del sistema, no una característica previa del terreno.",
        "Fase del proyecto (operando, cancelado...): describe lo que pasó con la inversión, no el sitio.",
    ],
})
excl

In [ ]:
rho = df[["capacity", "optimal_tilt", "pv_potential", "solar_aptitude"]].corr("spearman")["solar_aptitude"].drop("solar_aptitude")
rho.round(3).rename("ρ con solar_aptitude").to_frame()

El objetivo del modelo es **evaluar un sitio antes de construir**. Por eso solo se admiten características del terreno, el clima y la ubicación. Estas cuatro variables se conocen después de decidir la inversión y se excluyen por coherencia con el caso de uso. `optimal_tilt` merece una mención especial: en una iteración previa de la segunda entrega, usarla en lugar de `slope`, `aspect` y `curvature` hizo caer el recall de Baja casi a 0.

## 8.3 Identificadores y variables descriptivas

- **`OBJECTID`, `code` y `plant_name`** identifican cada planta y no aportan información predictiva generalizable.
- **`country`** (183 niveles) y **`size`** no forman parte de las 17 predictoras. `country` lo resumen las coordenadas (capítulo 5), y `size` deriva de `area`.

## 8.4 Conjunto final de predictoras

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

X = df[NUMERIC + CATEGORICAL]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("esc", StandardScaler())]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])
Xt = pre.fit_transform(X_train)
print(f"Predictoras originales: {X.shape[1]} ({len(NUMERIC)} numéricas + {len(CATEGORICAL)} categóricas)")
print(f"Columnas tras el preprocesamiento: {Xt.shape[1]}")
print(f"Entrenamiento: {X_train.shape[0]:,} filas  |  Prueba: {X_test.shape[0]:,} filas")

Se obtiene el mismo vector de **39 columnas** de la segunda entrega: 13 numéricas más 26 binarias del one-hot. El preprocesamiento se ajusta **solo con el conjunto de entrenamiento**, dentro del pipeline, para que la mediana de imputación y los parámetros de escalado no se filtren desde el conjunto de prueba.